# Phase 2 — Colab CUDA benchmark

Run this notebook before the full experiment. It verifies the GPU, installs the bundled project, creates the required zero-shot predictions, and compares ListMLE microbatch sizes 4, 8, and 16 on the same validation-only job. **It does not evaluate test positions.**

Before running: choose **Runtime → Change runtime type → GPU**. Select an **L4** if Colab offers one; this 8M-parameter model does not require an A100.

In [ ]:
import shutil
import subprocess
import sys

import torch

if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU is attached. Change the Colab runtime type to GPU, then rerun.")
subprocess.run(["nvidia-smi"], check=True)
print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
if shutil.which("colab"):
    print("\nCurrent Colab balance and burn rate, when exposed by this runtime:")
    subprocess.run(["colab", "usage"], check=False)

## Mount Drive and install the project

The first run prompts you to upload `esm2-mutation-effects-colab.zip`. The notebook copies it to Drive. Code runs from Colab's local disk for speed; results, checkpoints, the ESM-2 download, and the frozen-feature cache persist under `MyDrive/esm2-phase2-colab/`.

In [ ]:
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

from google.colab import drive, files

drive.mount("/content/drive")
PERSISTENT = Path("/content/drive/MyDrive/esm2-phase2-colab")
PERSISTENT.mkdir(parents=True, exist_ok=True)
BUNDLE_NAME = "esm2-mutation-effects-colab.zip"
BUNDLE = PERSISTENT / BUNDLE_NAME

if not BUNDLE.exists():
    sidebar_copy = Path("/content") / BUNDLE_NAME
    if sidebar_copy.exists():
        uploaded_bundle = sidebar_copy
    else:
        print(f"Choose {BUNDLE_NAME} in the upload dialog.")
        uploaded = files.upload()
        matches = [Path("/content") / name for name in uploaded if name.endswith(".zip")]
        if len(matches) != 1:
            raise RuntimeError("Upload exactly one project zip bundle.")
        uploaded_bundle = matches[0]
    shutil.copy2(uploaded_bundle, BUNDLE)
    print(f"Saved bundle to {BUNDLE}")
else:
    print(f"Reusing bundle from {BUNDLE}")

PROJECT = Path("/content/esm2-mutation-effects")
if PROJECT.exists():
    shutil.rmtree(PROJECT)
with zipfile.ZipFile(BUNDLE) as archive:
    archive.extractall("/content")
if not (PROJECT / "pyproject.toml").exists():
    raise RuntimeError("The uploaded zip is not the expected Colab project bundle.")

def persistent_directory(local_path, drive_path):
    drive_path.mkdir(parents=True, exist_ok=True)
    if local_path.is_symlink() or local_path.is_file():
        local_path.unlink()
    elif local_path.exists():
        shutil.rmtree(local_path)
    local_path.parent.mkdir(parents=True, exist_ok=True)
    local_path.symlink_to(drive_path, target_is_directory=True)

persistent_directory(PROJECT / "results", PERSISTENT / "results")
persistent_directory(PROJECT / "data" / "processed", PERSISTENT / "processed")
os.environ["TORCH_HOME"] = str(PERSISTENT / "torch-cache")
Path(os.environ["TORCH_HOME"]).mkdir(parents=True, exist_ok=True)
os.chdir(PROJECT)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[dev]"], check=True)
print(f"Project ready at {PROJECT}")
print(f"Persistent artifacts: {PERSISTENT}")

## Validate data and prepare zero-shot scores

The zero-shot file is generated once and then reused from Drive. This also downloads and caches the public `esm2_t6_8M_UR50D` weights.

In [ ]:
subprocess.run([sys.executable, "scripts/validate_dataset.py"], cwd=PROJECT, check=True)
zero_shot = PROJECT / "results" / "esm2_masked_marginal.csv"
if zero_shot.exists():
    print(f"Reusing {zero_shot}")
else:
    subprocess.run(
        [sys.executable, "scripts/score_zero_shot.py", "--device", "cuda", "--batch-size", "32"],
        cwd=PROJECT,
        check=True,
    )

## Benchmark CUDA microbatches

This runs one epoch per candidate using identical labels and an effective batch of 16 lists. It is a throughput check, not a scientific result. Failed out-of-memory candidates are recorded and skipped.

In [ ]:
benchmark_root = PROJECT / "results" / "phase2_cuda_microbatch_benchmark"
subprocess.run(
    [
        sys.executable,
        "scripts/benchmark_lora_list_batching.py",
        "--device",
        "cuda",
        "--output-root",
        str(benchmark_root),
    ],
    cwd=PROJECT,
    check=True,
)

In [ ]:
import json

import pandas as pd

summary_path = benchmark_root / "benchmark_summary.json"
benchmark = json.loads(summary_path.read_text())
display(pd.DataFrame(benchmark["results"]))
chosen = benchmark["fastest_completed_microbatch"]
if chosen is None:
    raise RuntimeError("No microbatch candidate completed; inspect the output above.")
print(f"\nSelected CUDA microbatch: {chosen}")
print(f"Saved benchmark: {summary_path}")

## Optional compute-unit check

Colab's live resource panel is the authoritative place to read the current compute-unit burn rate. Enter that number below to see how many GPU hours 100 units buys at the current rate. The rate can change, so this is a budget guardrail rather than a promise.

In [ ]:
COMPUTE_UNITS_PER_HOUR = None  # Example: replace None with the live rate shown by Colab
PURCHASED_UNITS = 100

completed = [row for row in benchmark["results"] if row["status"] == "completed"]
benchmark_hours = sum(float(row["epoch_seconds"]) for row in completed) / 3600
print(f"Measured benchmark GPU time: {benchmark_hours:.3f} hours")
if COMPUTE_UNITS_PER_HOUR is not None:
    available_hours = PURCHASED_UNITS / COMPUTE_UNITS_PER_HOUR
    print(f"At {COMPUTE_UNITS_PER_HOUR:g} units/hour, 100 units buys about {available_hours:.1f} GPU hours.")
    print(f"This benchmark used approximately {benchmark_hours * COMPUTE_UNITS_PER_HOUR:.2f} units.")
else:
    print("Set COMPUTE_UNITS_PER_HOUR from Colab's live Resources panel to calculate the budget.")

Benchmark complete. Next, open `02_run_phase2.ipynb`. Do not delete `MyDrive/esm2-phase2-colab/`; it contains the restart-safe state.